# 01 - Data Cleaning
### Google Play Store App Analytics

**Objective:** Inspect the raw Google Play Store dataset, identify data-quality
issues, and build a reproducible cleaning pipeline using Pandas.

Raw source: Kaggle "Google Play Store Apps" dataset (lava18), ~10,841 rows,
13 columns.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

RAW_PATH = Path('../data/raw/googleplaystore.csv')
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head()

(10841, 13)


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## 1. Inspect the raw dataset

In [2]:
print("Shape:", df.shape)
print()
print("Data types:")
print(df.dtypes)

Shape: (10841, 13)

Data types:
App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs              str
Type                  str
Price                 str
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object


In [3]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64


In [4]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate app names:", df['App'].duplicated().sum())

Fully duplicate rows: 483
Duplicate app names: 1181


In [5]:
for col in ['Category', 'Type', 'Content Rating']:
    print(col, '->', df[col].unique()[:15])
    print()

Category -> <StringArray>
[     'ART_AND_DESIGN',   'AUTO_AND_VEHICLES',              'BEAUTY', 'BOOKS_AND_REFERENCE',            'BUSINESS',              'COMICS',
       'COMMUNICATION',              'DATING',           'EDUCATION',       'ENTERTAINMENT',              'EVENTS',             'FINANCE',
      'FOOD_AND_DRINK',  'HEALTH_AND_FITNESS',      'HOUSE_AND_HOME']
Length: 15, dtype: str

Type -> <StringArray>
['Free', 'Paid', nan, '0']
Length: 4, dtype: str

Content Rating -> <StringArray>
['Everyone', 'Teen', 'Everyone 10+', 'Mature 17+', 'Adults only 18+', 'Unrated', nan]
Length: 7, dtype: str



## 2. Identify invalid / inconsistent values

The dataset is **not clean**. Key issues found:

- `Installs` stored as strings like `"10,000+"` -- needs numeric parsing.
- `Price` stored as strings like `"$4.99"` -- needs the `$` stripped.
- `Size` mixes `"19M"`, `"14k"`, and `"Varies with device"`.
- `Reviews` is stored as string/object, not integer.
- `Rating` has ~1,474 missing values, and at least one row has a rating
  outside the valid [0, 5] range due to a **column-shift bug**.
- `Last Updated` is a free-text date string.


In [6]:
# The famous shifted row: Category becomes a numeric string ('1.9'),
# which is actually the Rating value that shifted left by one column.
suspicious = df[pd.to_numeric(df['Category'], errors='coerce').notna()]
suspicious

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


In [7]:
print(df['Installs'].unique()[:10])
print(df['Price'].unique()[:10])
print(df['Size'].unique()[:10])

<StringArray>
['10,000+', '500,000+', '5,000,000+', '50,000,000+', '100,000+', '50,000+', '1,000,000+', '10,000,000+', '5,000+', '100,000,000+']
Length: 10, dtype: str
<StringArray>
['0', '$4.99', '$3.99', '$6.99', '$1.49', '$2.99', '$7.99', '$5.99', '$3.49', '$1.99']
Length: 10, dtype: str
<StringArray>
['19M', '14M', '8.7M', '25M', '2.8M', '5.6M', '29M', '33M', '3.1M', '28M']
Length: 10, dtype: str


In [8]:
print(df[df['Rating'] > 5])

                                           App Category  Rating Reviews    Size Installs Type     Price Content Rating             Genres  \
10472  Life Made WI-Fi Touchscreen Photo Frame      1.9    19.0    3.0M  1,000+     Free    0  Everyone            NaN  February 11, 2018   

      Last Updated Current Ver Android Ver  
10472       1.0.19  4.0 and up         NaN  


## 3. Run the cleaning pipeline

All transformation logic lives in `src/data_cleaning.py` so it is reusable,
testable, and reproducible outside the notebook. Each transformation is
explained in that module's docstrings and inline comments.

Transformations applied:
- **Shifted row repair**: rows where `Category` is a stray numeric value are
  detected generically and every field shifted back into its correct column;
  the row is then dropped since `Category` itself is unrecoverable.
- **Deduplication**: exact duplicate rows dropped, then duplicate `App` names
  collapsed to the entry with the most reviews (proxy for most complete/recent).
- **Installs**: `"10,000+"` -> `10000` (comma and `+` stripped, cast to int).
- **Price**: `"$4.99"` -> `4.99` (float).
- **Size**: `"19M"` -> `19.0` MB, `"14k"` -> `0.0137` MB, `"Varies with device"` -> `NaN`.
- **Rating**: bounded to [0, 5]; anything outside is treated as invalid -> `NaN`.
- **Type**: missing/invalid values inferred from `Price` (price > 0 implies Paid).
- **Derived columns**: `installs_numeric`, `price_numeric`, `size_mb`, `is_free`,
  `log_installs`, `log_reviews`, `review_to_install_ratio`, `updated_year`,
  `updated_month`, `install_bucket`, `rating_bucket`, `price_bucket`.


In [9]:
from data_cleaning import load_raw, clean_dataset

raw = load_raw()
cleaned = clean_dataset(raw)
print("Raw shape:    ", raw.shape)
print("Cleaned shape:", cleaned.shape)
cleaned.head()

Raw shape:     (10841, 13)
Cleaned shape: (9659, 23)


,app_id,app,category,rating,reviews,size_mb,installs,type,price,content_rating,genres,last_updated,current_version,android_version,is_free,log_installs,log_reviews,review_to_install_ratio,updated_year,updated_month,install_bucket,rating_bucket,price_bucket
0,1,Facebook,SOCIAL,4.1,78158306,NaN,1000000000,Free,0.0,Teen,Social,2018-08-03,Varies with device,Varies with device,1,20.723266,18.174247,0.078158,2018,8,10M+,4-4.5,Free
1,2,WhatsApp Messenger,COMMUNICATION,4.4,69119316,NaN,1000000000,Free,0.0,Everyone,Communication,2018-08-03,Varies with device,Varies with device,1,20.723266,18.051345,0.069119,2018,8,10M+,4-4.5,Free
2,3,Instagram,SOCIAL,4.5,66577446,NaN,1000000000,Free,0.0,Teen,Social,2018-07-31,Varies with device,Varies with device,1,20.723266,18.013876,0.066577,2018,7,10M+,4-4.5,Free
3,4,Messenger – Text and Video Chat for Free,COMMUNICATION,4.0,56646578,NaN,1000000000,Free,0.0,Everyone,Communication,2018-08-01,Varies with device,Varies with device,1,20.723266,17.852342,0.056647,2018,8,10M+,3-4,Free
4,5,Clash of Clans,GAME,4.6,44893888,98.0,100000000,Free,0.0,Everyone 10+,Strategy,2018-07-15,10.322.16,4.1 and up,1,18.420681,17.619812,0.448939,2018,7,10M+,4.5-5,Free


## 4. Validate the cleaned dataset

In [10]:
print("Nulls after cleaning:")
print(cleaned.isnull().sum())

Nulls after cleaning:
app_id                        0
app                           0
category                      0
rating                     1463
reviews                       0
size_mb                    1228
installs                      0
type                          0
price                         0
content_rating                0
genres                        0
last_updated                  0
current_version               8
android_version               2
is_free                       0
log_installs                  0
log_reviews                   0
review_to_install_ratio      15
updated_year                  0
updated_month                 0
install_bucket                0
rating_bucket              1463
price_bucket                  0
dtype: int64


In [11]:
print("Rating range:", cleaned['rating'].min(), '-', cleaned['rating'].max())
print("Installs range:", cleaned['installs'].min(), '-', cleaned['installs'].max())
print("Unique categories:", cleaned['category'].nunique())
print("Duplicate app names remaining:", cleaned['app'].duplicated().sum())

Rating range: 1.0 - 5.0
Installs range: 0 - 1000000000
Unique categories: 33
Duplicate app names remaining: 0


In [12]:
cleaned.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
app_id,9659.0,NaN,NaN,NaN,4830.0,1.0,2415.5,4830.0,7244.5,9659.0,2788.457459
app,9659,9659,Facebook,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,9659,33,FAMILY,1876,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,8196.0,NaN,NaN,NaN,4.173267,1.0,4.0,4.3,4.5,5.0,0.536253
reviews,9659.0,NaN,NaN,NaN,216804.110363,0.0,25.0,969.0,29453.5,78158306.0,1831430.21321
size_mb,8431.0,NaN,NaN,NaN,20.398075,0.008301,4.6,12.0,28.0,100.0,21.828959
installs,9659.0,<NA>,<NA>,<NA>,7798170.248162,0.0,1000.0,100000.0,1000000.0,1000000000.0,53769728.818071
type,9659,2,Free,8905,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,9659.0,NaN,NaN,NaN,1.097231,0.0,0.0,0.0,0.0,400.0,16.851618
content_rating,9659,6,Everyone,7903,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Save the cleaned dataset

Saved to `data/processed/googleplaystore_cleaned.csv` for use in the EDA
and business-analysis notebooks, and for loading into MySQL.

In [13]:
OUT_PATH = Path('../data/processed/googleplaystore_cleaned.csv')
cleaned.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, "| rows:", len(cleaned))

Saved:

 ../data/processed/googleplaystore_cleaned.csv | rows: 9659
